# Portfolio Optimization using QAOA from Scratch
### What is done in this section:
This section introduces the notebook. We declare the intent to solve the Portfolio Optimization problem entirely from scratch using the Quantum Approximate Optimization Algorithm (QAOA).



## 1. Mathematical Formulation
### What is done in this section:
We explain the mathematical formulation of the Portfolio Optimization problem.
We use the Markowitz Mean-Variance framework. Let $\mu$ be the expected return of assets, and $\Sigma$ the covariance matrix.
The objective function to minimize is:
$$ \min_{x \in \{0,1\}^n} rac{q}{2} x^T \Sigma x - \mu^T x $$
where $q$ is the risk factor.



## 2. QUBO and Ising Mapping
### What is done in this section:
We explain how to map the QUBO to an Ising Hamiltonian.
For variables $x_i \in \{0, 1\}$, we substitute $x_i = rac{1 - Z_i}{2}$ where $Z_i \in \{1, -1\}$ corresponds to the Pauli-Z matrix.
This allows us to write the cost function as a quantum observable (Cost Hamiltonian).



## 3. Libraries and Packages
### What is done in this section:
We import all the necessary classical and quantum simulation libraries.
We rely on `numpy` for classical math, `scipy.optimize` for the classical loop, and `qiskit` for circuit construction and statevector simulation.



In [ ]:
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.circuit import Parameter


## 4. Generate Mock Financial Data
### What is done in this section:
We create a synthetic dataset of 3 assets to keep the simulation tractable. We define their expected returns and a positive semi-definite covariance matrix.



In [ ]:
# Number of assets
n_assets = 3

# Expected returns
mu = np.array([0.1, 0.2, 0.15])

# Covariance matrix (must be symmetric positive semi-definite)
Sigma = np.array([
    [0.05, 0.01, 0.02],
    [0.01, 0.06, 0.03],
    [0.02, 0.03, 0.04]
])

# Risk factor
q = 0.5


## 5. Build Ising Hamiltonian
### What is done in this section:
We build the Cost Hamiltonian based on the formula: $rac{q}{2} (rac{1-Z}{2})^T \Sigma (rac{1-Z}{2}) - \mu^T (rac{1-Z}{2})$.
We ignore constant terms as they don't affect optimization, and collect the coefficients for single $Z_i$ and $Z_i Z_j$ interactions.



In [ ]:
# Build the Pauli operators for the cost Hamiltonian
linear_z = np.zeros(n_assets)
quadratic_zz = np.zeros((n_assets, n_assets))

for i in range(n_assets):
    linear_z[i] += mu[i] * 0.5
    for j in range(n_assets):
        Q_ij = q * Sigma[i, j] / 2
        linear_z[i] -= Q_ij * 0.5 * 2
        quadratic_zz[i, j] += Q_ij * 0.25

obs_terms = []
for i in range(n_assets):
    if abs(linear_z[i]) > 1e-8:
        pauli_str = ['I'] * n_assets
        pauli_str[n_assets - 1 - i] = 'Z'
        obs_terms.append(("".join(pauli_str), linear_z[i]))

for i in range(n_assets):
    for j in range(i+1, n_assets):
        coef = quadratic_zz[i, j] + quadratic_zz[j, i]
        if abs(coef) > 1e-8:
            pauli_str = ['I'] * n_assets
            pauli_str[n_assets - 1 - i] = 'Z'
            pauli_str[n_assets - 1 - j] = 'Z'
            obs_terms.append(("".join(pauli_str), coef))

cost_hamiltonian = SparsePauliOp.from_list(obs_terms)
print("Cost Hamiltonian:\n", cost_hamiltonian)


## 6. Introduction to QAOA
### What is done in this section:
We introduce the QAOA layers framework. We need p layers of alternating Cost ($H_C$) and Mixer ($H_M$) evolution.



## 7. Mixer Hamiltonian
### What is done in this section:
We define the mixer Hamiltonian $H_M = \sum_i X_i$. It creates superpositions and explores the solution space.



In [ ]:
# Mixer Hamiltonian consists of Pauli X on each qubit
mixer_terms = []
for i in range(n_assets):
    pauli_str = ['I'] * n_assets
    pauli_str[n_assets - 1 - i] = 'X'
    mixer_terms.append(("".join(pauli_str), 1.0))

mixer_hamiltonian = SparsePauliOp.from_list(mixer_terms)
print("Mixer Hamiltonian:\n", mixer_hamiltonian)


## 8. Parameterized QAOA Circuit
### What is done in this section:
We build a parameterized Qiskit `QuantumCircuit` representation of the QAOA ansatz.



In [ ]:
def create_qaoa_circuit(p):
    circ = QuantumCircuit(n_assets)
    for i in range(n_assets):
        circ.h(i)
        
    gamma_params = [Parameter(f"gamma_{k}") for k in range(p)]
    beta_params = [Parameter(f"beta_{k}") for k in range(p)]
    
    for k in range(p):
        for pauli, coef in cost_hamiltonian.to_list():
            val = coef.real
            z_idx = [i for i, c in enumerate(reversed(pauli)) if c == 'Z']
            if len(z_idx) == 1:
                circ.rz(2 * gamma_params[k] * val, z_idx[0])
            elif len(z_idx) == 2:
                circ.cx(z_idx[0], z_idx[1])
                circ.rz(2 * gamma_params[k] * val, z_idx[1])
                circ.cx(z_idx[0], z_idx[1])
        
        for i in range(n_assets):
            circ.rx(2 * beta_params[k], i)
            
    return circ, gamma_params, beta_params

p_layers = 2
qaoa_circ, gammas, betas = create_qaoa_circuit(p_layers)


## 9. Expectation Value Calculation
### What is done in this section:
We define an objective function that evaluates the expectation value $\langle \psi | H_C | \psi angle$.



In [ ]:
def expectation_value(params):
    p = len(params) // 2
    gamma_vals = params[:p]
    beta_vals = params[p:]
    
    bind_dict = {gammas[i]: gamma_vals[i] for i in range(p)}
    bind_dict.update({betas[i]: beta_vals[i] for i in range(p)})
        
    bound_circ = qaoa_circ.assign_parameters(bind_dict)
    sv = Statevector.from_instruction(bound_circ)
    return sv.expectation_value(cost_hamiltonian).real


## 10. Classical Optimizer Setup
### What is done in this section:
We prepare random starting bounds to pass to our classical loop `COBYLA`.



In [ ]:
initial_params = np.random.rand(2 * p_layers) * np.pi
print("Initial Expectation Value:", expectation_value(initial_params))


## 11. Run QAOA (Optimization Loop)
### What is done in this section:
We execute the optimization loop and retrieve the optimal parameters.



In [ ]:
res = opt.minimize(expectation_value, initial_params, method='COBYLA', options={'maxiter': 200})
print("Optimization Result:\n", res)
optimal_params = res.x
